In [ ]:
##OLD VERSION##
import numpy as  np
from numba import njit
from itertools import product

OPEN = (6, 7, 8)

def actions_from_roll(d1, d2, d3, d4, OPEN):
    """ Returns up to 3 unique actions (dx,dy,dz) from the 3 pairings.
    OPEN must be of length 3. """
    o0, o1, o2 = OPEN
    pairings = ((d1+d2, d3+d4), (d1+d3, d2+d4), (d1+d4, d2+d3))
    acts = set()
    for s1, s2 in pairings:
        dx = (1 if s1 == 6 else 0) + (1 if s2 == 6 else 0)
        dy = (1 if s1 == 7 else 0) + (1 if s2 == 7 else 0)
        dz = (1 if s1 == 8 else 0) + (1 if s2 == 8 else 0)
        if dx + dy + dz > 0:
            acts.add((dx, dy, dz))
    return list(acts)

def precompute_actions(OPEN):
    """ Builds a padded action array of shape (1296, 3, 3) with -1 (because numba requires fixed size arrays). """
    rolls = list(product(range(1, 7), repeat=4))  # 6^4 = 1296 possible rolls with 4 dice
    R = len(rolls)
    Amax = 3
    actions = -np.ones((R, Amax, 3), dtype=np.int8) # size = 1296*3*3

    for r, (d1, d2, d3, d4) in enumerate(rolls):
        acts = actions_from_roll(d1, d2, d3, d4, OPEN)
        for k in range(min(len(acts), Amax)):
            actions[r, k, 0] = acts[k][0]
            actions[r, k, 1] = acts[k][1]
            actions[r, k, 2] = acts[k][2]

    return actions

@njit
def compute_V(Nx, Ny, Nz, actions):
    
    V = np.zeros((Nx + 1, Ny + 1, Nz + 1))
    V[Nx, Ny, Nz] = Nx + Ny + Nz

    Smax = Nx + Ny + Nz
    R = actions.shape[0]
    Amax = actions.shape[1]

    for s in range(Smax - 1, -1, -1):
        # iterates all (x,y,z) with x+y+z = s within bounds
        
        # Nx
        x_min = max(0, s - (Ny + Nz)) # y+z = s-x 
        x_max = min(Nx, s)
        for x in range(x_min, x_max + 1):
            dif = s - x
            y_min = max(0, dif - Nz)
            y_max = min(Ny, dif)
            
            # Ny
            for y in range(y_min, y_max + 1):
                z = dif - y
                if z < 0 or z > Nz:
                    continue
                if x == Nx and y == Ny and z == Nz:
                    continue

                immediate_gain = x + y + z  # if we stop

                # if we keep going
                future_gain = 0.0
                for r in range(R):
                    best = 0.0  # if no admissible action, we get 0
                    for k in range(Amax):
                        dx = actions[r, k, 0]
                        if dx < 0:
                            continue
                        dy = actions[r, k, 1]
                        dz = actions[r, k, 2]
                        nx = x + dx
                        ny = y + dy
                        nz = z + dz
                        if nx <= Nx and ny <= Ny and nz <= Nz: # checks admissibility
                            val = V[nx, ny, nz]
                            if val > best:
                                best = val
                    future_gain += 1/R * best

                V[x, y, z] = immediate_gain if immediate_gain >= future_gain else future_gain

    return V



In [56]:
N6, N7, N8 = 100, 100, 100
OPEN = (6, 7, 8)
actions = precompute_actions(OPEN)
V = compute_V(N6, N7, N8, actions)
print("Le gain optimal est :", V[0, 0, 0])

Le gain optimal est : 6.330720105660973


In [ ]:
## NEW VERSION WITH ACTIONS INSTEAD OF ROLLS ##
import numpy as np
from numba import njit

OPEN = (6, 7, 8)

def compute_probabilities():
    pairs = {}
    for dice1 in range(1,7):
        for dice2 in range(1,7):
            for dice3 in range(1,7):
                for dice4 in range(1,7):
                    couples_get = (
                        tuple(sorted([dice1 + dice2, dice3 + dice4])),
                        tuple(sorted([dice1 + dice3, dice2 + dice4])),
                        tuple(sorted([dice1 + dice4, dice2 + dice3]))
                    )

                    if couples_get not in pairs:
                        pairs[couples_get] = 1/(6**4)
                    else:
                        pairs[couples_get] += 1/(6**4)

    pair_keys = list(pairs.keys())
    R = len(pair_keys)

    pair_array = np.zeros((R,3,2), dtype=np.int8)
    prob_array = np.zeros(R)

    for i,k in enumerate(pair_keys):
        for j in range(3):
            pair_array[i,j,0] = k[j][0]
            pair_array[i,j,1] = k[j][1]
        prob_array[i] = pairs[k]
    #print(pair_array[22])
    return pair_array, prob_array


def actions_from_roll(pairings, OPEN):
    o0, o1, o2 = OPEN
    acts = set()
    for p1, p2 in pairings:
        dx = (1 if p1 == o0 else 0) + (1 if p2 == o0 else 0)
        dy = (1 if p1 == o1 else 0) + (1 if p2 == o1 else 0)
        dz = (1 if p1 == o2 else 0) + (1 if p2 == o2 else 0)

        if dx + dy + dz > 0:
            acts.add((dx, dy, dz))

    return list(acts)


def precompute_actions(OPEN):
    pair_array, prob_array = compute_probabilities()
    R = pair_array.shape[0]
    Amax = 3
    actions = -np.ones((R, Amax, 3), dtype=np.int8)
    for r in range(R):
        pairings = (
            tuple(pair_array[r,0]),
            tuple(pair_array[r,1]),
            tuple(pair_array[r,2])
        )
        acts = actions_from_roll(pairings, OPEN)
        if r==22:
            print(pairings, acts)

        for k in range(min(len(acts), Amax)):
            actions[r,k] = acts[k]
    print(actions[0])
    return actions, pair_array, prob_array


@njit
def compute_V(Nx, Ny, Nz, actions, prob_array):

    V = np.zeros((Nx + 1, Ny + 1, Nz + 1))
    V[Nx, Ny, Nz] = Nx + Ny + Nz

    Smax = Nx + Ny + Nz
    R = actions.shape[0]
    Amax = actions.shape[1]
    policy = np.zeros((Nx + 1, Ny + 1, Nz + 1, 3), dtype=np.int8)

    for s in range(Smax - 1, -1, -1):
        x_min = max(0, s - (Ny + Nz))
        x_max = min(Nx, s)

        for x in range(x_min, x_max + 1):
            dif = s - x
            y_min = max(0, dif - Nz)
            y_max = min(Ny, dif)

            for y in range(y_min, y_max + 1):
                z = dif - y
                if z < 0 or z > Nz:
                    continue
                if x == Nx and y == Ny and z == Nz:
                    continue
                immediate_gain = x + y + z
                future_gain = 0.0
                for r in range(R):
                    best = 0.0
                    for a in range(Amax):
                        dx = actions[r,a,0]
                        if dx < 0:
                            continue
                        dy = actions[r,a,1]
                        dz = actions[r,a,2]
                        nx = x + dx
                        ny = y + dy
                        nz = z + dz
                        val = V[min(nx, Nx), min(ny, Ny), min(nz, Nz)]
                        if val > best:
                            #policy[x, y, z] = (x+dx, y+dy, z+dz)
                            best = val
                    future_gain += prob_array[r] * best
                if immediate_gain >= future_gain:
                    V[x,y,z] = immediate_gain
                    #policy[x, y, z] = (x, y, z)
                else:
                    V[x,y,z] = future_gain

    return V, policy

In [8]:
N6, N7, N8 = 100, 100, 100
OPEN = (6, 7, 8)
actions, pair_array, prob_array = precompute_actions(OPEN)
V, policy = compute_V(N6, N7, N8, actions, prob_array)
print("Le gain optimal est :", V[0, 0, 0])

[[-1 -1 -1]
 [-1 -1 -1]
 [-1 -1 -1]]
Le gain optimal est : 6.330720105661309


In [5]:
N6, N7, N8 = 10, 10, 10
OPEN = (6, 7, 8)
actions, pair_array, prob_array = precompute_actions(OPEN)
V, policy = compute_V(N6, N7, N8, actions, prob_array)
print("Le gain optimal est :", V[0, 0, 0])

Le gain optimal est : 6.328950820370662


In [72]:
def simulation_lancers(N=100):
    lancers = []
    for i in range(1, N+1):
        d1, d2, d3, d4 = np.random.randint(1, 7, size=4, dtype=int)
        lancers.append((d1, d2, d3, d4))
    return lancers
print(simulation_lancers(10))

[(np.int64(6), np.int64(6), np.int64(5), np.int64(6)), (np.int64(4), np.int64(3), np.int64(3), np.int64(4)), (np.int64(6), np.int64(6), np.int64(1), np.int64(6)), (np.int64(5), np.int64(3), np.int64(4), np.int64(2)), (np.int64(6), np.int64(2), np.int64(4), np.int64(1)), (np.int64(4), np.int64(3), np.int64(1), np.int64(1)), (np.int64(6), np.int64(5), np.int64(2), np.int64(6)), (np.int64(1), np.int64(2), np.int64(6), np.int64(4)), (np.int64(2), np.int64(2), np.int64(5), np.int64(2)), (np.int64(6), np.int64(6), np.int64(1), np.int64(2))]


In [ ]:
def simulation_player(lancers, policy, actions, pair_array): # always continue
    nx, ny, nz = 0, 0, 0
    for lancer in lancers:
        d1, d2, d3, d4 = lancer
        pairings = (
            tuple(sorted([d1 + d2, d3 + d4])),
            tuple(sorted([d1 + d3, d2 + d4])),
            tuple(sorted([d1 + d4, d2 + d3]))
        )
        pairings = np.array(pairings, dtype=np.int8)
        r = np.bincount(np.where(np.all(pair_array == pairings, axis=1))[0]).argmax()
        acts = actions[r]
        nb_fail = 0
        for a in range(acts.shape[0]):
            if acts[a,0] < 0:
                nb_fail += 1
            else:
                nx, ny, nz = (nx, ny, nz) + acts[a]
                print("Player moves to:", (nx, ny, nz))
        if nb_fail == acts.shape[0]:
            print(lancer)
            print("No possible action, player loses")
            return 0,0,0
        if nx == policy[nx, ny, nz, 0] and ny == policy[nx, ny, nz, 1] and nz == policy[nx, ny, nz, 2]:
            print("Player decides to stop at:", (nx, ny, nz))
            return nx, ny, nz

    return nx, ny, nz

In [75]:
lancer = simulation_lancers(20)
V, policy = compute_V(N6, N7, N8, actions, prob_array)


In [91]:
simulation_player(lancer, policy, actions, pair_array)

Player moves to: (np.int64(1), np.int64(0), np.int64(0))
Player moves to: (np.int64(1), np.int64(1), np.int64(0))
Player moves to: (np.int64(1), np.int64(1), np.int64(1))
Player moves to: (np.int64(1), np.int64(2), np.int64(1))
Player moves to: (np.int64(1), np.int64(3), np.int64(1))
(np.int64(2), np.int64(2), np.int64(1), np.int64(2))
No possible action, player loses


(0, 0, 0)

In [77]:
print(pair_array[168])
actions[154]

[[ 7 10]
 [ 6 11]
 [ 6 11]]


array([[ 0,  1,  1],
       [-1, -1, -1],
       [-1, -1, -1]], dtype=int8)

In [53]:
precompute_actions(OPEN)

((np.int8(3), np.int8(4)), (np.int8(2), np.int8(5)), (np.int8(3), np.int8(4))) []
[[-1 -1 -1]
 [-1 -1 -1]
 [-1 -1 -1]]


(array([[[-1, -1, -1],
         [-1, -1, -1],
         [-1, -1, -1]],
 
        [[-1, -1, -1],
         [-1, -1, -1],
         [-1, -1, -1]],
 
        [[-1, -1, -1],
         [-1, -1, -1],
         [-1, -1, -1]],
 
        ...,
 
        [[-1, -1, -1],
         [-1, -1, -1],
         [-1, -1, -1]],
 
        [[-1, -1, -1],
         [-1, -1, -1],
         [-1, -1, -1]],
 
        [[-1, -1, -1],
         [-1, -1, -1],
         [-1, -1, -1]]], shape=(336, 3, 3), dtype=int8),
 array([[[ 2,  2],
         [ 2,  2],
         [ 2,  2]],
 
        [[ 2,  3],
         [ 2,  3],
         [ 2,  3]],
 
        [[ 2,  4],
         [ 2,  4],
         [ 2,  4]],
 
        ...,
 
        [[11, 11],
         [11, 11],
         [10, 12]],
 
        [[11, 12],
         [11, 12],
         [11, 12]],
 
        [[12, 12],
         [12, 12],
         [12, 12]]], shape=(336, 3, 2), dtype=int8),
 array([0.0007716 , 0.00308642, 0.00308642, 0.00308642, 0.00308642,
        0.00308642, 0.00154321, 0.00308642, 0.00